# AlphaShot - Mechanics Simulation with Alpha Particle Impacts

This notebook simulates the rotational motion of a vane driven by alpha particle impacts within a gas environment. It imports the `Vane` class from `src.mechanics.py`, the `AlphaParticle` class from `src.particles.py`, and the `GasEnvironment` class from `src.gas.py`. The simulation tracks the momentum transfer from alpha particles to the vane, resulting in its rotation.

**Important Considerations and Future Improvements:**
*   **Collision Model:** The current impact model incorporates a coefficient of restitution for more realistic collisions.
*   **Parameter Exploration:** Interactive widgets using `ipywidgets` will be added to allow for real-time parameter adjustments, including gas selection.
*   **Reproducibility:** The simulation is made reproducible by seeding the random number generator.
*   **Data Saving:** Results will be saved to a CSV file in the `data` directory for later analysis.
*   **Performance:** For longer simulations, the code will be profiled to identify and address performance bottlenecks.


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from src.mechanics import Vane
from src.particles import AlphaParticle
from src.gas import GasEnvironment
import numpy.fft as fft
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# Simulation parameters
np.random.seed(42)  # Seed for reproducibility

moment_of_inertia = 1e-6  # kg*m^2
width = 0.1  # m
height = 0.05  # m
damping_coefficient = 1e-8  # N*m*s/rad
dt = 0.001  # s (Time step)
duration = 10.0  # s (Simulation duration)
impact_rate = 1000  # Impacts per second
alpha_kinetic_energy_MeV = 5.0 # MeV

gas_pressure=101325.0 # Pascals
gas_temperature=293.15 # Kelvin

# Gas selection widget
gas_options = ["Hydrogen", "Deuterium", "Helium", "Nitrogen", "Argon", "Xenon"]
gas_dropdown = widgets.Dropdown(options=gas_options, value="Hydrogen", description="Gas Type:")
display(gas_dropdown)

# Create a Gas Environment (initially Hydrogen, but will be updated by the widget)
gas_environment = GasEnvironment(gas_type=gas_dropdown.value, pressure=gas_pressure, temperature=gas_temperature)
mean_free_path = gas_environment.calculate_mean_free_path()
print(f"Mean Free Path: {mean_free_path:.4e} m")


In [ ]:
# Create a Vane object
vane = Vane(moment_of_inertia=moment_of_inertia, width=width, height=height, damping_coefficient=damping_coefficient)

In [ ]:
# Run the simulation
time = np.arange(0, duration, dt)
angle_history = []
angular_velocity_history = []
torque_history = [] # Initialize torque history
impact_count = 0 # Count the number of impacts

def run_simulation(gas_type):
    global angle_history, angular_velocity_history, torque_history, impact_count, final_angle, gas_environment, mean_free_path
    
    # Update gas environment based on widget selection
    gas_environment = GasEnvironment(gas_type=gas_type, pressure=gas_pressure, temperature=gas_temperature)
    mean_free_path = gas_environment.calculate_mean_free_path()
    print(f"Mean Free Path: {mean_free_path:.4e} m")
    
    angle_history = []
    angular_velocity_history = []
    torque_history = []
    impact_count = 0

    for t in time:
        # Simulate alpha particle impacts
        if np.random.random() < impact_rate * dt:  # Probability of impact during this time step
            # Create an alpha particle with a random initial velocity direction
            theta = np.random.uniform(0, 2 * np.pi) # Random angle for the initial velocity vector
            initial_velocity = np.array([np.cos(theta), np.sin(theta), 0.0])  # Direction vector
            alpha = AlphaParticle(initial_position=[0.0, 0.0, 0.0], kinetic_energy_MeV=alpha_kinetic_energy_MeV, initial_velocity_vector=initial_velocity)
            torque = vane.apply_particle_impact(alpha, dt) # Get the torque from the impact
            impact_count += 1
        else:
            torque = 0.0 # No impact, no torque

        angle, angular_velocity, _ = vane.get_state()
        angle_history.append(angle)
        angular_velocity_history.append(angular_velocity)
        torque_history.append(torque) # Store the torque

    final_angle = angle  # Store the final angle

# Observe the gas_dropdown widget to re-run the simulation when the gas type changes
def on_gas_change(change):
    run_simulation(change['new'])

gas_dropdown.observe(on_gas_change, names='value')

# Run the simulation initially with the default gas
run_simulation(gas_dropdown.value)

In [ ]:
# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(time, angle_history)
plt.xlabel("Time (s)")
plt.ylabel("Angle (rad)")
plt.title("Vane Angle Over Time with Alpha Particle Impacts")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(time, angular_velocity_history)
plt.xlabel("Time (s)")
plt.ylabel("Angular Velocity (rad/s)")
plt.title("Vane Angular Velocity Over Time with Alpha Particle Impacts")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(time, torque_history)
plt.xlabel("Time (s)")
plt.ylabel("Torque (N*m)")
plt.title("Torque Applied to Vane Over Time")
plt.grid(True)
plt.show()

# FFT analysis of angular velocity
yf = fft.fft(angular_velocity_history)
T = dt  # Sample spacing
xf = fft.fftfreq(len(angular_velocity_history), T)[:len(angular_velocity_history)//2]
plt.figure(figsize=(10, 6))
plt.plot(xf, np.abs(yf[0:len(angular_velocity_history)//2]))
plt.xlabel("Frequency (Hz)")
plt.ylabel("FFT Magnitude")
plt.title("FFT of Vane Angular Velocity")
plt.grid(True)
plt.show()

# Histogram of Angular Velocities
plt.figure(figsize=(10, 6))
plt.hist(angular_velocity_history, bins=50)
plt.xlabel("Angular Velocity (rad/s)")
plt.ylabel("Frequency")
plt.title("Histogram of Vane Angular Velocities")
plt.grid(True)
plt.show()

# Print key simulation results
print(f"Final Angle: {final_angle:.4f} rad")
print(f"Number of Impacts: {impact_count}")
print(f"Final Angle ($\theta$): {final_angle:.4f} rad") # LaTeX inline
print(f"Number of Impacts: {impact_count}")


In [ ]:
# Save the results to a CSV file (example)
import pandas as pd #Uncomment if you want to run this
results = pd.DataFrame({"time": time, "angle": angle_history, "angular_velocity": angular_velocity_history, "torque": torque_history})
results.to_csv("data/simulation_results.csv", index=False)